In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Exploratory Data Analysis - Euclid Photometry\n",
    "\n",
    "This notebook analyzes the Euclid photometric dataset before training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from pathlib import Path\n",
    "\n",
    "from config import cfg\n",
    "from data.dataset import find_measurements, filter_and_clean_data\n",
    "from utils.helpers import set_global_seed, setup_logging\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "set_global_seed(cfg.seed)\n",
    "logger = setup_logging()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data\n",
    "df = pd.read_csv(cfg.input_csv)\n",
    "print(f\"Raw data: {len(df)} rows, {len(df.columns)} columns\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "missing = df.isnull().sum()\n",
    "missing_percent = (missing / len(df)) * 100\n",
    "missing_df = pd.DataFrame({\n",
    "    'Missing Count': missing,\n",
    "    'Missing %': missing_percent\n",
    "}).sort_values('Missing %', ascending=False)\n",
    "\n",
    "print(f\"Columns with > 50% missing values: {len(missing_df[missing_df['Missing %'] > 50])}\")\n",
    "missing_df.head(20)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Filter and clean data\n",
    "df_clean = filter_and_clean_data(cfg.input_csv, cfg.cleaned_csv)\n",
    "print(f\"Clean data: {len(df_clean)} rows, {len(df_clean.columns)} columns\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Find measurements\n",
    "measurements = find_measurements(df_clean)\n",
    "print(f\"Found {len(measurements)} measurements\")\n",
    "\n",
    "# Show measurement details\n",
    "measurement_df = pd.DataFrame(measurements)\n",
    "measurement_df[['band', 'survey', 'type', 'wavelength']].head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize wavelength coverage\n",
    "fig, ax = plt.subplots(figsize=(14, 6))\n",
    "\n",
    "bands = [m['band'] for m in measurements]\n",
    "wavelengths = [m['wavelength'] for m in measurements]\n",
    "surveys = [m['survey'] for m in measurements]\n",
    "\n",
    "# Color by survey\n",
    "survey_colors = {'euclid': 'red', 'decam': 'blue', 'panstarrs': 'green', \n",
    "                 'hsc': 'orange', 'lsst': 'purple', 'megacam': 'brown', 'jpcam': 'pink'}\n",
    "colors = [survey_colors.get(s, 'gray') for s in surveys]\n",
    "\n",
    "scatter = ax.scatter(wavelengths, range(len(wavelengths)), \n",
    "                    c=colors, s=50, alpha=0.7)\n",
    "ax.set_xlabel('Wavelength (Å)', fontsize=12)\n",
    "ax.set_ylabel('Measurement Index', fontsize=12)\n",
    "ax.set_title(f'Wavelength Coverage - {len(measurements)} Measurements', fontsize=14)\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "# Create legend\n",
    "from matplotlib.patches import Patch\n",
    "legend_elements = [Patch(facecolor='red', label='Euclid'),\n",
    "                   Patch(facecolor='blue', label='DECam'),\n",
    "                   Patch(facecolor='green', label='Pan-STARRS'),\n",
    "                   Patch(facecolor='orange', label='HSC'),\n",
    "                   Patch(facecolor='purple', label='LSST')]\n",
    "ax.legend(handles=legend_elements, loc='upper right')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('wavelength_coverage.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Survey distribution\n",
    "survey_counts = pd.Series(surveys).value_counts()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "survey_counts.plot(kind='bar', ax=ax, color='steelblue')\n",
    "ax.set_xlabel('Survey', fontsize=12)\n",
    "ax.set_ylabel('Number of Measurements', fontsize=12)\n",
    "ax.set_title('Survey Distribution', fontsize=14)\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('survey_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check data statistics for a sample\n",
    "from data.dataset import build_measurement_dataset_arcsinh\n",
    "\n",
    "# Sample 1000 rows\n",
    "df_sample = df_clean.sample(n=min(1000, len(df_clean)), random_state=cfg.seed)\n",
    "values, mask, band, survey, type_ids, _, _, _ = build_measurement_dataset_arcsinh(df_sample, measurements)\n",
    "\n",
    "print(f\"Values shape: {values.shape}\")\n",
    "print(f\"Mask shape: {mask.shape}\")\n",
    "print(f\"Valid measurements per sample: {mask.sum(axis=1).mean():.2f}\")\n",
    "print(f\"Min valid: {mask.sum(axis=1).min()}, Max valid: {mask.sum(axis=1).max()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize flux distribution (arcsinh transformed)\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "# Get valid flux values\n",
    "valid_flux = values[mask][:, 0]\n",
    "valid_error = values[mask][:, 1]\n",
    "\n",
    "axes[0].hist(valid_flux, bins=100, color='steelblue', alpha=0.7, edgecolor='black')\n",
    "axes[0].set_xlabel('Flux (arcsinh)', fontsize=12)\n",
    "axes[0].set_ylabel('Frequency', fontsize=12)\n",
    "axes[0].set_title(f'Flux Distribution (n={len(valid_flux):,})', fontsize=12)\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "axes[1].hist(valid_error, bins=100, color='coral', alpha=0.7, edgecolor='black')\n",
    "axes[1].set_xlabel('Error (arcsinh)', fontsize=12)\n",
    "axes[1].set_ylabel('Frequency', fontsize=12)\n",
    "axes[1].set_title(f'Error Distribution (n={len(valid_error):,})', fontsize=12)\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('flux_error_distributions.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check correlation between flux and error\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "ax.scatter(valid_flux, valid_error, alpha=0.1, s=1, c='steelblue')\n",
    "ax.set_xlabel('Flux (arcsinh)', fontsize=12)\n",
    "ax.set_ylabel('Error (arcsinh)', fontsize=12)\n",
    "ax.set_title(f'Flux vs Error Correlation (r={np.corrcoef(valid_flux, valid_error)[0,1]:.3f})', fontsize=12)\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('flux_vs_error.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary statistics\n",
    "print(\"=\"*60)\n",
    "print(\"DATASET SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Total rows: {len(df_clean):,}\")\n",
    "print(f\"Total columns: {len(df_clean.columns)}\")\n",
    "print(f\"Total measurements: {len(measurements)}\")\n",
    "print(f\"Unique bands: {len(set([m['band'] for m in measurements]))}\")\n",
    "print(f\"Unique surveys: {len(set([m['survey'] for m in measurements]))}\")\n",
    "print(f\"Unique types: {len(set([m['type'] for m in measurements]))}\")\n",
    "print(f\"\\nValid measurements per object:\")\n",
    "print(f\"  - Mean: {mask.sum(axis=1).mean():.2f}\")\n",
    "print(f\"  - Std: {mask.sum(axis=1).std():.2f}\")\n",
    "print(f\"  - Min: {mask.sum(axis=1).min()}\")\n",
    "print(f\"  - Max: {mask.sum(axis=1).max()}\")\n",
    "print(f\"\\nFlux statistics (arcsinh):\")\n",
    "print(f\"  - Mean: {valid_flux.mean():.4f}\")\n",
    "print(f\"  - Std: {valid_flux.std():.4f}\")\n",
    "print(f\"  - Min: {valid_flux.min():.4f}\")\n",
    "print(f\"  - Max: {valid_flux.max():.4f}\")\n",
    "print(f\"\\nError statistics (arcsinh):\")\n",
    "print(f\"  - Mean: {valid_error.mean():.4f}\")\n",
    "print(f\"  - Std: {valid_error.std():.4f}\")\n",
    "print(f\"  - Min: {valid_error.min():.4f}\")\n",
    "print(f\"  - Max: {valid_error.max():.4f}\")\n",
    "print(\"=\"*60)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}